In [ ]:
!pip install dotenv langchain_openai

In [10]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage
from langgraph.graph.message import MessagesState
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

load_dotenv()

# 初始化模型
model = init_chat_model(
    "qwen3.7-plus-2026-05-26",
    model_provider="openai",
    temperature=0.5,
    max_tokens=1024,
    timeout=60,
    max_retries=3,
    base_url=os.getenv("DASHSCOPE_API_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    extra_body={
        "thinking": {
            "type": "disabled"
            }
    }
)

# 创建Agent，传入模型和工具列表
# agent = create_agent(
#     model = model,
#     tools = [get_weather, calculate],
#     system_prompt = "你是一个乐于助人的助手，会使用工具来回答问题"
# )

# 1. 定义状态
class OverAllState(MessagesState):
    username: str
    output: str

# 2. 定义节点
def node_a(state: OverAllState) -> OverAllState:
    '''
    向全局状态添加messages
    messages 中包含 HumanMessage 类型的消息
    '''
    return {
        "messages": [HumanMessage("你好，我是"+state["username"])]
    }

def llm_node(state: OverAllState) -> OverAllState:
    '''
    调用模型，生成回复
    '''
    result = model.invoke(state["messages"])
    return {
        "messages": [result],
        "output": result.content
    }

# 3. 构建状态图
builder = StateGraph(state_schema=OverAllState)

builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

# 4. 运行状态图
result = graph.invoke({"username": "张三"})
print(result)

{'messages': [HumanMessage(content='你好，我是张三', additional_kwargs={}, response_metadata={}, id='59625eaa-f917-4e75-9b83-f9a17608ea33'), AIMessage(content='你好，张三！很高兴认识你。请问今天有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 16, 'total_tokens': 32, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'text_tokens': 16}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus-2026-05-26', 'system_fingerprint': None, 'id': 'chatcmpl-b00fe65f-1d97-99a2-979e-fd8b5f0410d1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019febc3-9e4a-7d10-888e-1a2fed207a2c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 16, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})], 'username': '张三', 'output': '你好，张三！很高兴认识你。请问今天有什么我可以帮你的吗？'}
